In [1]:
from unsloth import FastLanguageModel

def load_model(model_name: str, max_sequence_length: int = 2048, load_in_4bit: bool = True):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_sequence_length,
        load_in_4bit = load_in_4bit,  #
        # token = "YOUR_HF_TOKEN", # HF Token for gated models
    )
    return model, tokenizer


model, tokenizer = load_model("unsloth/mistral-7b-instruct-v0.3-bnb-4bit")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/anaconda/envs/finetune_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-16GB. Num GPUs = 1. Max memory: 15.773 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
# === Verify: no double BOS in training or inference prompts ===
import sys; sys.path.insert(0, ".")
from finetuning.prompts import build_training_text, build_inference_prompt

# 1) Check inference prompt text — should NOT start with <s>
inf_text = build_inference_prompt(tokenizer, "Hallo wereld")
print("Inference prompt starts with <s>?", inf_text.startswith(tokenizer.bos_token))
print("Inference prompt (first 80 chars):", repr(inf_text[:80]))

# 2) Tokenize the way inference.py does (add_special_tokens=True, the default)
inf_tokens = tokenizer(inf_text, add_special_tokens=True)
first_tokens = tokenizer.convert_ids_to_tokens(inf_tokens["input_ids"][:8])
print("Inference token IDs (first 8):", inf_tokens["input_ids"][:8])
print("Inference tokens   (first 8):", first_tokens)
assert first_tokens.count("<s>") == 1, f"DOUBLE BOS in inference! {first_tokens}"

# 3) Check training text — same BOS guarantee
train_text = build_training_text(tokenizer, "Hallo wereld", "Dit is het antwoord.")
print("\nTraining text starts with <s>?", train_text.startswith(tokenizer.bos_token))
train_tokens = tokenizer(train_text, add_special_tokens=True)
first_train = tokenizer.convert_ids_to_tokens(train_tokens["input_ids"][:8])
print("Training token IDs (first 8):", train_tokens["input_ids"][:8])
print("Training tokens   (first 8):", first_train)
assert first_train.count("<s>") == 1, f"DOUBLE BOS in training! {first_train}"

print("\n✅ Single BOS confirmed for both training and inference")

Inference prompt starts with <s>? False
Inference prompt (first 80 chars): '[INST] Beantwoord de volgende vraag zo goed mogelijk in het Nederlands.\n\nHallo w'
Inference token IDs (first 8): [1, 3, 2507, 1208, 1577, 1324, 1108, 2976]
Inference tokens   (first 8): ['<s>', '[INST]', '▁Be', 'ant', 'wo', 'ord', '▁de', '▁vol']

Training text starts with <s>? False
Training token IDs (first 8): [1, 3, 2507, 1208, 1577, 1324, 1108, 2976]
Training tokens   (first 8): ['<s>', '[INST]', '▁Be', 'ant', 'wo', 'ord', '▁de', '▁vol']

✅ Single BOS confirmed for both training and inference


In [4]:
user_question = "Beantwoord de volgende vraag zo goed mogelijk in het Nederlands. Adviseer iemand hoe te ontstressen. Antwoord in 3 bullet points"

messages = [
    {"role": "user", "content": user_question},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")



outputs = model.generate(
    inputs,
    max_new_tokens=600,                  # hard stop
    temperature=0.1,
    eos_token_id=tokenizer.eos_token_id, # stop on EOS
    pad_token_id=tokenizer.pad_token_id,
    #use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<s>[INST] Beantwoord de volgende vraag zo goed mogelijk in het Nederlands. Adviseer iemand hoe te ontstressen. Antwoord in 3 bullet points[/INST] 1. Lichamelijke activiteit: Doe regelmatig lichamelijke activiteit, zoals wandelen, joggen of yoga. Dit stimuleert de productie van endorfines, de zogenoemde 'gelukshormoon'.

2. Relaxatie: Probeer om regelmatig te rusten en te ontspannen. Je kan hiervoor mediteren, luisteren naar rustige muziek, schrijven in een dagboek of een hobby uitvoeren.

3. Sociale contacten: Versterk je sociale banden door met vrienden en familie te praten, samen te eten of samen te doen. Sociale contacten kunnen je voelen van stress helpen en je gevoel van welzijn vergroten.</s>


In [13]:

user_question = "Explain AI in 2 phrases"

messages = [
    {"role": "user", "content": user_question},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")



outputs = model.generate(
    inputs,
    max_new_tokens=600,                  # hard stop
    eos_token_id=tokenizer.eos_token_id, # stop on EOS
    #pad_token_id=tokenizer.pad_token_id,
    #use_cache=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))

<s>[INST] Explain AI in 2 phrases[/INST] 1. AI is a branch of computer science that enables machines to mimic intelligent human behavior.

2. AI uses algorithms and statistical models to analyze data, make decisions, and solve complex problems.</s>


In [ ]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = OpenAI(  
  base_url = "https://finetuning-foundry.openai.azure.com/openai/v1/",  
  api_key=token_provider,
)

response = client.chat.completions.create(
  model="grok-4-fast-reasoning",  # Replace with your model deployment name.
    messages=[
        {"role": "system", "content": "Assistant is a large language model trained by OpenAI."},
        {"role": "user", "content": "Who were the founders of Microsoft?"}
    ]
)

#print(response)
print(response.model_dump_json(indent=2))
print(response.choices[0].message.content)

In [5]:
from datasets import load_from_disk

alpaca_train_formatted = load_from_disk("datasets/alpaca_train_formatted", keep_in_memory=True)

In [7]:
alpaca_test_formatted = load_from_disk("datasets/alpaca_test_formatted", keep_in_memory=True)

In [8]:
alpaca_train_formatted[0]

{'text': 'Beantwoord de volgende vraag zo goed mogelijk.\n\n### Vraag:\nSchrijf een gedicht van 7 regels met behulp van de gegeven woorden\ninput: paraplu, rail, water, lucht, gras, blad\n\n### Antwoord:\nOnder de donker wordende lucht,\nHet gras groeit nog steeds.\nEen eenzaam blad drijft weg\nVan de grijze ijzeren rail,\nDobberend als een paraplu in de golven van water.\nLangzaam dringt de nacht binnen, brengt rust en kalmte.\nEn ik sta en adem, voel de vrede in deze balsem.\n</s>'}